# Notebook 10: Explanatory — Education Momentum and Reintegration

## Section 1 — Problem Framing

**Business question:** Among residents who have been in the program, which education
trajectory patterns — attendance trend, progress acceleration, consistency — are most
strongly associated with completing reintegration?

**Who cares:** Program directors deciding how much to invest in education services, and
social workers designing Individual Service Plans (ISPs) for each resident.

**Approach: Explanatory.** We use logistic regression (via `statsmodels`) so that each
odds ratio has a direct interpretation. A Random Forest classifier provides a
predictive comparison.

**Success metric:** Pseudo-R2 (McFadden), coefficient significance, and odds-ratio
interpretation in business terms.

## Section 2 — Data Acquisition and Preparation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, accuracy_score
from sklearn.ensemble import RandomForestClassifier
import statsmodels.api as sm
import json, os, warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

RESULTS_DIR = '../data/explanatory_results/'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup complete.')

In [ ]:
DATA = '../backend/HarborOfHope.API/Data/lighthouse_csv_v7/'

residents = pd.read_csv(DATA + 'residents.csv')
education = pd.read_csv(DATA + 'education_records.csv')
health    = pd.read_csv(DATA + 'health_wellbeing_records.csv')
sessions  = pd.read_csv(DATA + 'process_recordings.csv')

print(f'Loaded {len(residents)} residents, {len(education)} education records, '
      f'{len(health)} health records, {len(sessions)} sessions')

In [ ]:
def parse_duration_months(s):
    """Convert '15 Years 9 months' to numeric months."""
    if pd.isna(s):
        return np.nan
    try:
        parts = str(s).split()
        return int(parts[0]) * 12 + int(parts[2])
    except Exception:
        return np.nan

def to_bool_int(series):
    return series.map({True: 1, False: 0, 'True': 1, 'False': 0}).fillna(0).astype(int)

def plot_coefficients(params, conf_int, pvalues, title, filename):
    df_coef = pd.DataFrame({
        'coef': params, 'ci_low': conf_int.iloc[:, 0],
        'ci_high': conf_int.iloc[:, 1], 'pvalue': pvalues
    })
    df_coef = df_coef.drop('const', errors='ignore')
    df_coef['significant'] = df_coef['pvalue'] < 0.05
    df_coef = df_coef.sort_values('coef')
    fig, ax = plt.subplots(figsize=(10, max(6, len(df_coef) * 0.35)))
    colors = ['#2196F3' if s else '#BDBDBD' for s in df_coef['significant']]
    y_pos = range(len(df_coef))
    ax.barh(y_pos, df_coef['coef'], color=colors, edgecolor='white', height=0.7)
    ax.errorbar(df_coef['coef'], y_pos,
                xerr=[df_coef['coef'] - df_coef['ci_low'], df_coef['ci_high'] - df_coef['coef']],
                fmt='none', ecolor='black', capsize=3, linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_coef.index, fontsize=9)
    ax.axvline(0, color='red', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Coefficient')
    ax.set_title(title)
    blue_patch = plt.Line2D([0], [0], color='#2196F3', lw=6, label='p < 0.05')
    grey_patch = plt.Line2D([0], [0], color='#BDBDBD', lw=6, label='p >= 0.05')
    ax.legend(handles=[blue_patch, grey_patch], loc='lower right')
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions defined.')

In [ ]:
edu_sorted = education.sort_values(['resident_id', 'record_date'])

def calc_slope(group, col):
    vals = group[col].dropna().values
    if len(vals) < 2:
        return 0.0
    x = np.arange(len(vals))
    return np.polyfit(x, vals, 1)[0]

edu_traj = edu_sorted.groupby('resident_id').apply(
    lambda g: pd.Series({
        'attendance_slope': calc_slope(g, 'attendance_rate'),
        'progress_slope': calc_slope(g, 'progress_percent'),
        'avg_attendance': g['attendance_rate'].mean(),
        'avg_progress': g['progress_percent'].mean(),
        'max_progress': g['progress_percent'].max(),
        'attendance_std': g['attendance_rate'].std(),
        'edu_months': len(g),
    })
).reset_index()

health_avg = health.groupby('resident_id').agg(
    avg_health=('general_health_score', 'mean'),
    avg_nutrition=('nutrition_score', 'mean')
).reset_index()

sess_count = sessions.groupby('resident_id')['recording_id'].count().reset_index()
sess_count.columns = ['resident_id', 'total_sessions']

r2 = residents.copy()
r2['age_months'] = r2['age_upon_admission'].apply(parse_duration_months)
r2['stay_months'] = r2['length_of_stay'].apply(parse_duration_months)

r2_bool = [c for c in r2.columns if c.startswith('sub_cat_') or c in ['is_pwd', 'has_special_needs']]
for col in r2_bool:
    r2[col] = to_bool_int(r2[col])
r2['abuse_types_count'] = r2[[c for c in r2.columns if c.startswith('sub_cat_')]].sum(axis=1)
r2['family_risk_count'] = r2[['family_solo_parent', 'family_indigenous',
                               'family_parent_pwd', 'family_informal_settler']].apply(
    lambda row: sum(to_bool_int(pd.Series(row))), axis=1)

df2 = r2[['resident_id', 'reintegration_status', 'age_months', 'stay_months',
           'abuse_types_count', 'has_special_needs', 'family_risk_count']].copy()
df2 = df2.merge(edu_traj, on='resident_id', how='left')
df2 = df2.merge(health_avg, on='resident_id', how='left')
df2 = df2.merge(sess_count, on='resident_id', how='left')
df2['total_sessions'] = df2['total_sessions'].fillna(0)
df2['attendance_std'] = df2['attendance_std'].fillna(0)

df2['reint_completed'] = (df2['reintegration_status'] == 'Completed').astype(int)
df2 = df2.dropna(subset=['avg_attendance', 'avg_health', 'age_months', 'stay_months'])

print(f'Pipeline 2 analytical dataset: {df2.shape[0]} rows x {df2.shape[1]} columns')
print(f'\nReintegration outcome:\n{df2["reint_completed"].value_counts().rename({0:"Not Completed",1:"Completed"})}')

## Section 3 — Exploration

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(data=df2, x='reint_completed', palette=['#EF5350', '#66BB6A'], ax=axes[0, 0])
axes[0, 0].set_xticklabels(['Not Completed', 'Completed'])
axes[0, 0].set_title('Reintegration Outcome Distribution')

sns.boxplot(data=df2, x='reint_completed', y='attendance_slope', palette=['#EF5350', '#66BB6A'], ax=axes[0, 1])
axes[0, 1].set_xticklabels(['Not Completed', 'Completed'])
axes[0, 1].set_title('Attendance Slope by Outcome')

sns.boxplot(data=df2, x='reint_completed', y='avg_progress', palette=['#EF5350', '#66BB6A'], ax=axes[1, 0])
axes[1, 0].set_xticklabels(['Not Completed', 'Completed'])
axes[1, 0].set_title('Avg Education Progress by Outcome')

sns.scatterplot(data=df2, x='avg_attendance', y='avg_progress', hue='reint_completed',
                palette=['#EF5350', '#66BB6A'], alpha=0.7, ax=axes[1, 1])
axes[1, 1].set_title('Attendance vs Progress (colored by outcome)')
axes[1, 1].legend(title='Completed', labels=['No', 'Yes'])

plt.suptitle('Pipeline 2 — Education Trajectory & Reintegration', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('p2_eda.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4 — Modeling and Feature Selection

In [ ]:
feature_cols_2 = ['attendance_slope', 'progress_slope', 'avg_attendance', 'avg_progress',
                  'max_progress', 'attendance_std', 'edu_months', 'avg_health',
                  'avg_nutrition', 'total_sessions', 'age_months', 'stay_months',
                  'abuse_types_count', 'has_special_needs', 'family_risk_count']
X2 = df2[feature_cols_2].copy()
y2 = df2['reint_completed'].copy()

scaler2 = StandardScaler()
X2_scaled = pd.DataFrame(scaler2.fit_transform(X2), columns=X2.columns, index=X2.index)

X2_sm = sm.add_constant(X2_scaled)
logit_model_2 = sm.Logit(y2, X2_sm).fit(disp=0)
print(logit_model_2.summary())

print('\n=== Odds Ratios ===')
odds = np.exp(logit_model_2.params).drop('const')
print(odds.sort_values(ascending=False).to_string())

In [ ]:
plot_coefficients(logit_model_2.params, logit_model_2.conf_int(), logit_model_2.pvalues,
                  'Pipeline 2 — Logistic Regression Coefficients (log-odds, standardized)',
                  'p2_logit_coefficients.png')

In [ ]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2_scaled, y2, test_size=0.2, random_state=42, stratify=y2)

rf2 = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42, class_weight='balanced')
rf2.fit(X2_train, y2_train)
y2_pred = rf2.predict(X2_test)
y2_proba = rf2.predict_proba(X2_test)[:, 1]

print('Random Forest Classification Report:')
print(classification_report(y2_test, y2_pred, target_names=['Not Completed', 'Completed']))

cv2 = cross_val_score(rf2, X2_scaled, y2, cv=5, scoring='accuracy')
print(f'5-Fold CV Accuracy: {cv2.mean():.3f} +/- {cv2.std():.3f}')

fpr, tpr, _ = roc_curve(y2_test, y2_proba)
auc_val = roc_auc_score(y2_test, y2_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color='#2196F3', lw=2, label=f'RF AUC = {auc_val:.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Pipeline 2 — ROC Curve (Random Forest)')
ax.legend()
plt.tight_layout()
plt.savefig('p2_roc.png', dpi=150, bbox_inches='tight')
plt.show()

fi2 = pd.Series(rf2.feature_importances_, index=X2.columns).sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
fi2.plot.barh(color='#66BB6A', ax=ax)
ax.set_title('Pipeline 2 — Random Forest Feature Importance')
plt.tight_layout()
plt.savefig('p2_rf_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 5 — Evaluation and Causal Analysis

### Key Findings
The logistic regression reveals which education-trajectory features are most strongly
associated with the *log-odds* of completing reintegration, after controlling for
resident demographics and service intensity.

### Interpreting Odds Ratios
- An odds ratio > 1 means the feature is associated with *higher* odds of completion.
- An odds ratio < 1 means it is associated with *lower* odds.

### Causal Defensibility
- **Education engagement -> reintegration**: Theoretically defensible. Education provides
  structure, life skills, and self-efficacy. However, *selection bias* remains: residents
  on a better trajectory may be more likely to be assessed as "ready."
- **Length of stay**: Longer stay may associate with completion simply because there is
  more time to complete the process. We include it as a control, not as a causal lever.
- **Health scores**: Better health may enable participation, or may reflect that
  well-managed residents improve on both health and reintegration simultaneously.

### Limitations
- Small sample size (N ~ 60) limits statistical power and generalizability.
- Binary outcome collapses meaningful variation ("In Progress" vs. "On Hold").
- No instrumental variable or regression discontinuity available.

### Recommendations
1. **Monitor attendance slope monthly** — a sustained negative slope is an early warning.
2. **Set minimum thresholds** — e.g., residents with avg_attendance < 0.65 should
   trigger additional support.
3. **Invest in education consistency** (low attendance_std) over raw attendance level.

## Section 6 — Data Leakage Check

### Potential Leakage Concerns
1. **Reintegration status as endpoint**: We use `reintegration_status == 'Completed'` as
   the target. Features are measured *during* the stay, so temporal ordering is partially
   maintained — education records occur before the reintegration decision.
   **Mitigation**: We only use features from the education/health/session tables that
   represent the resident's trajectory during their stay, not post-reintegration data.
2. **Length of stay**: Could leak if longer stays mechanically lead to completion (e.g.,
   the process requires N months). **Mitigation**: We include it as a control variable
   and note that its coefficient should be interpreted with caution.
3. **No future data leakage**: All features (attendance, progress, health, sessions) are
   recorded during the program and do not peek into the future.

### Verdict
No direct leakage detected. The main concern is selection bias (healthier/better-
performing residents may be selected for reintegration), which we acknowledge as a
limitation of the observational design.

## Section 7 — Deployment Notes

### Integration with Harbor of Hope Web Application
- **API Endpoint**: `GET /api/explanatoryinsights/2` returns the logistic regression
  coefficients, odds ratios, and business interpretations.
- **Dashboard Page**: Admin > Insights page shows the reintegration readiness factors.
- **Alert System**: Flag residents whose attendance slope turns negative for 2+
  consecutive months.
- **Notebook location**: `ml-pipelines/10-explanatory-education-reintegration.ipynb`

### How to Refresh Results
1. Run this notebook end-to-end.
2. The final cell exports updated results to
   `data/explanatory_results/pipeline_02_education_reintegration.json`.

In [ ]:
sig2 = logit_model_2.pvalues.drop('const', errors='ignore')
sig2 = sig2[sig2 < 0.05].sort_values()

results_02 = {
    "pipeline_id": 2,
    "pipeline_name": "Education Momentum -> Reintegration",
    "target_variable": "Reintegration Completed (binary)",
    "model_type": "Logistic Regression (statsmodels)",
    "pseudo_r_squared": round(float(logit_model_2.prsquared), 4),
    "sample_size": int(len(y2)),
    "significant_features": [
        {
            "name": feat,
            "coefficient": round(float(logit_model_2.params[feat]), 4),
            "odds_ratio": round(float(np.exp(logit_model_2.params[feat])), 4),
            "p_value": round(float(logit_model_2.pvalues[feat]), 4),
            "direction": "increases odds" if logit_model_2.params[feat] > 0 else "decreases odds"
        }
        for feat in sig2.index
    ]
}

with open(f'{RESULTS_DIR}pipeline_02_education_reintegration.json', 'w') as f:
    json.dump(results_02, f, indent=2)
print(f'Exported results to {RESULTS_DIR}pipeline_02_education_reintegration.json')